# 7 prompt anti-patterns and how to fix them

**The Agent Build Log · Build #016 · Prompt Engineering 8/8**

Seven habits that show up in real prompts, each measured against its fix on the same 20 support tickets
(build #014's test set, labelled before any model run), 3 runs per ticket. Some hurt a lot. Some, it turns out, don't.

1. [Setup](#1-setup)
2. [Vague task](#2-vague-task)
3. [Conflicting instructions](#3-conflicting-instructions)
4. [SHOUTING](#4-shouting)
5. [Only saying what not to do](#5-only-saying-what-not-to-do)
6. [A broken format example](#6-a-broken-format-example)
7. [Padding the prompt with unrelated context](#7-padding-the-prompt-with-unrelated-context)
8. [Trusting the model's confidence score](#8-trusting-the-models-confidence-score)
9. [Scorecard](#9-scorecard)

**Model:** Claude Haiku 5.5, thinking off. **Needs:** an Anthropic API key. A full run costs about **13 cents**.

In [1]:
import json, os, random, re
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
import anthropic

load_dotenv(Path.cwd() / ".env") or load_dotenv(Path.cwd().parents[1] / ".env")
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in a .env file (see README)"
pd.set_option("display.width", 170); pd.set_option("display.max_colwidth", 90)

MODEL = "claude-haiku-5-5"
PRICE = {"input": 0.10, "output": 0.50}
client = anthropic.Anthropic().with_options(max_retries=6)
TK = json.loads(Path("data/tickets.json").read_text())
TICKETS, FIELDS = TK["test"], TK["fields"]
_rub = Path("data/rubric_v3.md").read_text()
ROLE = re.search(r"^system: (.*)$", _rub, re.M).group(1)
RUBRIC = _rub.split("---\n", 2)[2].strip()   # build #014's v3, the current best prompt
SCHEMA = ('{"category": one of ' + json.dumps(FIELDS["category"]) + ', "priority": one of ' + json.dumps(FIELDS["priority"])
          + ', "sentiment": one of ' + json.dumps(FIELDS["sentiment"]) + "}")
print(f"{len(TICKETS)} tickets | model {MODEL}")

20 tickets | model claude-haiku-5-5


## 1. Setup

One helper runs a prompt over the 20 tickets, 3 runs each, 8 calls at a time. Two scorers:
- **strict JSON:** the reply must be valid JSON (`json.loads`), optionally inside code fences, with one allowed value per field.
- **words:** a free-text reply is usable only if each field has exactly one allowed value in it (as in build #009).

In [2]:
def run_prompt(system, make_user, max_tokens=400):
    jobs = [(t, r) for t in TICKETS for r in (1, 2, 3)]
    def one(j):
        t, run = j
        kw = {"system": system} if system else {}
        r = client.messages.create(model=MODEL, max_tokens=max_tokens, thinking={"type": "disabled"},
                                   messages=[{"role": "user", "content": make_user(t["text"])}], **kw)
        return {"id": t["id"], "run": run, "text": "".join(b.text for b in r.content if b.type == "text"),
                "input": r.usage.input_tokens, "output": r.usage.output_tokens,
                "cost": (r.usage.input_tokens * PRICE["input"] + r.usage.output_tokens * PRICE["output"]) / 1e6}
    with ThreadPoolExecutor(8) as ex:
        return pd.DataFrame(list(ex.map(one, jobs)))

LABEL = {t["id"]: t["label"] for t in TICKETS}

def strict_json(text):
    body = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip())
    try:
        d = json.loads(body)
    except json.JSONDecodeError:
        return None
    if not isinstance(d, dict):
        return None
    out = {f: str(d.get(f, "")).strip().lower() for f in FIELDS}
    return out if all(out[f] in FIELDS[f] for f in FIELDS) else None

def words(text):
    low = text.lower().replace("feature request", "feature_request")
    out = {}
    for f, allowed in FIELDS.items():
        hits = {v for v in allowed if re.search(rf"\b{re.escape(v)}\b", low)}
        if len(hits) != 1:
            return None
        out[f] = hits.pop()
    return out

def score_labels(df, parser):
    got = df["text"].map(parser)
    usable = got.notna()
    right = [g == LABEL[i] if g else False for g, i in zip(got, df["id"])]
    return {"usable %": round(100 * usable.mean()), "all 3 right %": round(100 * sum(right) / len(df)),
            "avg output tokens": round(df["output"].mean()), "$ per 1,000 tickets": round(df["cost"].mean() * 1000, 3)}

TRIAGE = lambda extra="": (lambda ticket: f"{RUBRIC}\n\nTriage this support ticket: give its category, priority and sentiment.\n\n"
                                          f"<ticket>\n{ticket}\n</ticket>\n\n{extra}")
CALM_FORMAT = f"Reply with only a JSON object, no other text:\n{SCHEMA}"
results = {}

## 2. Vague task

**Anti-pattern:** "Here's a support ticket. What do you think?" **Fix:** the rubric, a specific task and an output format (build #009).

In [3]:
vague = run_prompt(None, lambda t: f"Here's a support ticket. What do you think?\n\n{t}")
fixed = run_prompt(ROLE, TRIAGE(CALM_FORMAT))
results["1. vague task"] = {"anti-pattern": score_labels(vague, words), "fix": score_labels(fixed, strict_json)}
print("one vague reply:\n", vague.iloc[0]["text"][:500])
pd.DataFrame(results["1. vague task"]).T

one vague reply:
 This is a high-urgency ticket. A full office locked out with 40 people waiting means real work is stopped, so I'd triage it as P1 or Sev 1 right away.

It's also a good ticket in some ways, since it tells us when the problem started ("since the update") and how big the impact is. What it's missing:

- **Which system?** Email, a SaaS app, VPN, the domain login, or an internal app? "Log in" could mean several different things.
- **What exactly happens?** Is there an error message, a spinning wheel


,usable %,all 3 right %,avg output tokens,"$ per 1,000 tickets"
anti-pattern,2.0,0.0,362.0,0.185
fix,100.0,90.0,30.0,0.085


## 3. Conflicting instructions

Task: write the reply to the customer. **Anti-pattern:** the system prompt says *keep every reply under 50 words*, the
request says *explain clearly and in detail what we'll do and why, covering every step*. **Fix:** one consistent instruction.

In [4]:
REPLY_SYSTEM_SHORT = "You write replies to customers for an online electronics shop. Keep every reply under 50 words."
conflict = run_prompt(REPLY_SYSTEM_SHORT, lambda t: f"<ticket>\n{t}\n</ticket>\n\nWrite the reply to this customer. "
                      "Explain clearly and in detail what we will do and why, covering every step.")
consistent = run_prompt(REPLY_SYSTEM_SHORT, lambda t: f"<ticket>\n{t}\n</ticket>\n\nWrite the reply to this customer: "
                        "what we'll do next, in under 50 words.")

def reply_stats(df):
    n = df["text"].map(lambda s: len(s.split()))
    return {"under 50 words %": round(100 * (n < 50).mean()), "median words": int(n.median()), "max words": int(n.max()),
            "$ per 1,000 tickets": round(df["cost"].mean() * 1000, 3)}
results["2. conflicting instructions"] = {"anti-pattern": reply_stats(conflict), "fix": reply_stats(consistent)}
pd.DataFrame(results["2. conflicting instructions"]).T

,under 50 words %,median words,max words,"$ per 1,000 tickets"
anti-pattern,67.0,47.0,83.0,0.054
fix,82.0,43.0,56.0,0.050


## 4. SHOUTING

**Anti-pattern:** `IMPORTANT!!! You MUST ALWAYS reply with ONLY a JSON object!!! NEVER add ANY other text!!! THIS IS CRITICAL!!!`
**Fix:** the calm version, `Reply with only a JSON object, no other text`. Both with the same rubric.

In [5]:
SHOUT = (f"IMPORTANT!!! You MUST ALWAYS reply with ONLY a JSON object!!! NEVER add ANY other text!!! THIS IS CRITICAL!!!\n{SCHEMA}")
shout = run_prompt(ROLE, TRIAGE(SHOUT))
results["3. SHOUTING"] = {"anti-pattern": score_labels(shout, strict_json), "fix": score_labels(fixed, strict_json)}
pd.DataFrame(results["3. SHOUTING"]).T

,usable %,all 3 right %,avg output tokens,"$ per 1,000 tickets"
anti-pattern,100.0,85.0,30.0,0.089
fix,100.0,90.0,30.0,0.085


## 5. Only saying what not to do

Task: write a short reply to the customer. Markdown (`**bold**`, `# headings`, `- bullets`) shows up as literal symbols in
many support tools. Three versions: **no instruction**, **"Don't use markdown."**, and the **positive** version that says
what to write instead.

In [6]:
MD = re.compile(r"\*\*|__|^\s*#{1,6}\s|^\s*[-*•]\s|^\s*\d+\.\s", re.M)
REPLY_SYSTEM = "You write replies to customers for an online electronics shop."
base = lambda extra: (lambda t: f"<ticket>\n{t}\n</ticket>\n\nWrite a short reply to this customer.{extra}")
md_none = run_prompt(REPLY_SYSTEM, base(""))
md_dont = run_prompt(REPLY_SYSTEM, base(" Don't use markdown."))
md_positive = run_prompt(REPLY_SYSTEM, base(" Write it as one or two plain paragraphs, the way you'd type a plain-text email."))

def md_stats(df):
    return {"replies with markdown %": round(100 * df["text"].map(lambda s: bool(MD.search(s))).mean()),
            "median words": int(df["text"].map(lambda s: len(s.split())).median())}
results["4. only saying what not to do"] = {"no instruction": md_stats(md_none), "anti-pattern": md_stats(md_dont), "fix": md_stats(md_positive)}
pd.DataFrame(results["4. only saying what not to do"]).T

,replies with markdown %,median words
no instruction,20,89
anti-pattern,5,90
fix,0,78


## 6. A broken format example

**Anti-pattern:** an example of the output that isn't valid JSON (single quotes, a trailing comma), as people often
type it from memory. **Fix:** a valid example. Scored with a strict JSON parser.

In [7]:
BROKEN = ("Reply with only a JSON object, no other text, like this example:\n"
          "{'category': 'billing', 'priority': 'high', 'sentiment': 'neutral',}")
VALID = ("Reply with only a JSON object, no other text, like this example:\n"
         '{"category": "billing", "priority": "high", "sentiment": "neutral"}')
broken = run_prompt(ROLE, TRIAGE(BROKEN))
valid = run_prompt(ROLE, TRIAGE(VALID))
results["5. broken format example"] = {"anti-pattern": score_labels(broken, strict_json), "fix": score_labels(valid, strict_json)}
print("first reply with the broken example:", broken.iloc[0]["text"][:200])
pd.DataFrame(results["5. broken format example"]).T

first reply with the broken example: {'category': 'account', 'priority': 'urgent', 'sentiment': 'negative',}


,usable %,all 3 right %,avg output tokens,"$ per 1,000 tickets"
anti-pattern,25.0,23.0,33.0,0.082
fix,100.0,90.0,30.0,0.080


## 7. Padding the prompt with unrelated context

**Anti-pattern:** "give it everything, just in case": the 150-product catalog from build #006 pasted above the rubric
(about 10K tokens the task never needs). **Fix:** only what the task needs.

In [8]:
KINDS = ["Wireless Mouse", "Mechanical Keyboard", "USB-C Hub", "Laptop Stand", "Webcam", "Noise-Cancelling Headset",
         "Portable SSD", "Monitor Arm", "Docking Station", "Ergonomic Chair Cushion"]
BRANDS = ["Arcly", "Nimbus", "Voltra", "Kestrel", "Orbit"]
rng = random.Random(42); lines = []
for i in range(150):   # same catalog as build #006
    brand = rng.choice(BRANDS)
    lines.append(f"SKU-{1000 + i} | {brand} {KINDS[i % len(KINDS)]} {chr(65 + i % 26)}{i // 26 + 1} | ${round(rng.uniform(19, 349), 2)} | "
                 f"stock {rng.randint(0, 250)} | warranty {rng.choice([6, 12, 24, 36])} mo | returns {rng.choice([14, 30, 60])} days | "
                 f"works with {', '.join(rng.sample(['Windows', 'macOS', 'Linux', 'iPadOS', 'ChromeOS', 'Android'], 3))}")
CATALOG = "\n".join(lines)
padded = run_prompt(ROLE, lambda t: f"Product catalog, for reference:\n<catalog>\n{CATALOG}\n</catalog>\n\n" + TRIAGE(CALM_FORMAT)(t))
results["6. padding with unrelated context"] = {"anti-pattern": {**score_labels(padded, strict_json), "input tokens": round(padded["input"].mean())},
                                               "fix": {**score_labels(fixed, strict_json), "input tokens": round(fixed["input"].mean())}}
pd.DataFrame(results["6. padding with unrelated context"]).T

,usable %,all 3 right %,avg output tokens,"$ per 1,000 tickets",input tokens
anti-pattern,100.0,90.0,29.0,1.113,10983.0
fix,100.0,90.0,30.0,0.085,698.0


## 8. Trusting the model's confidence score

**Anti-pattern:** ask for a confidence score and route on it ("auto-accept above 90"). Does the number separate right
answers from wrong ones?

In [9]:
CONF = ('Reply with only a JSON object, no other text:\n{"category": ..., "priority": ..., "sentiment": ..., '
        '"confidence": a number from 0 to 100 for how sure you are that all three labels are right}\nAllowed values: ' + SCHEMA)
conf = run_prompt(ROLE, TRIAGE(CONF))

def conf_parse(text):
    body = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip())
    try:
        d = json.loads(body)
        return {f: str(d[f]).lower() for f in FIELDS}, float(d["confidence"])
    except Exception:
        return None, None

parsed = conf["text"].map(conf_parse)
conf["labels"] = parsed.map(lambda p: p[0]); conf["confidence"] = parsed.map(lambda p: p[1])
conf["right"] = [l == LABEL[i] if l else False for l, i in zip(conf["labels"], conf["id"])]
ok = conf[conf["confidence"].notna()]
cal = {"all 3 right %": round(100 * ok["right"].mean()),
       "avg confidence when right": round(ok[ok.right]["confidence"].mean(), 1),
       "avg confidence when wrong": round(ok[~ok.right]["confidence"].mean(), 1),
       "wrong answers with confidence >= 90": f"{int(((~ok.right) & (ok.confidence >= 90)).sum())}/{int((~ok.right).sum())}",
       "right answers with confidence >= 90": f"{int(((ok.right) & (ok.confidence >= 90)).sum())}/{int(ok.right.sum())}"}
results["7. trusting confidence"] = {"anti-pattern": cal}
print(ok["confidence"].value_counts().sort_index().to_dict())
pd.Series(cal)

{70.0: 1, 80.0: 8, 85.0: 12, 88.0: 17, 90.0: 3, 92.0: 1, 93.0: 1, 95.0: 10, 97.0: 3}


all 3 right %                             89
avg confidence when right               88.5
avg confidence when wrong               82.5
wrong answers with confidence >= 90      0/6
right answers with confidence >= 90    18/50
dtype: object

## 9. Scorecard

Every number below comes from the cells above; these are the numbers the LinkedIn post uses.

In [10]:
total = sum(df["cost"].sum() for df in [vague, fixed, conflict, consistent, shout, md_none, md_dont, md_positive, broken, valid, padded, conf])
f = {"model": MODEL, "sdk": anthropic.__version__, "thinking": "disabled", "results": results, "cost_total": round(total, 4),
     "confidence_counts": ok["confidence"].value_counts().sort_index().to_dict()}
Path("findings.json").write_text(json.dumps(f, indent=2, default=str))
print(json.dumps(f, indent=1, default=str))

{
 "model": "claude-haiku-5-5",
 "sdk": "1.15.0",
 "thinking": "disabled",
 "results": {
  "1. vague task": {
   "anti-pattern": {
    "usable %": 2,
    "all 3 right %": 0,
    "avg output tokens": 362,
    "$ per 1,000 tickets": 0.185
   },
   "fix": {
    "usable %": 100,
    "all 3 right %": 90,
    "avg output tokens": 30,
    "$ per 1,000 tickets": 0.085
   }
  },
  "2. conflicting instructions": {
   "anti-pattern": {
    "under 50 words %": 67,
    "median words": 47,
    "max words": 83,
    "$ per 1,000 tickets": 0.054
   },
   "fix": {
    "under 50 words %": 82,
    "median words": 43,
    "max words": 56,
    "$ per 1,000 tickets": 0.05
   }
  },
  "3. SHOUTING": {
   "anti-pattern": {
    "usable %": 100,
    "all 3 right %": 85,
    "avg output tokens": 30,
    "$ per 1,000 tickets": 0.089
   },
   "fix": {
    "usable %": 100,
    "all 3 right %": 90,
    "avg output tokens": 30,
    "$ per 1,000 tickets": 0.085
   }
  },
  "4. only saying what not to do": {
   "no inst